# 04c｜Kaggle 单卡实体立场 SFT 试验

使用 04b 导出的**本人私有**转运包及其 SHA256，承接 domain_v1 的 74 步权重。
首次跨平台仅导入模型权重并建立新的优化器；不是恢复 Colab 的优化器。
在 Kaggle 的 Add Input 添加**私有**数据集，包含 `130u_kaggle_transfer.zip`。
Session options 选择 GPU T4 x2 和 Internet on。本试验只使用第一块 GPU。

依次运行，记录第一格实际代码提交 SHA。以后跨会话恢复时若 main 已更新，
请将第一格的 `REF = "main"` 改为首次运行时的完整 40 位代码 SHA，再依次运行。
恢复只支持与原训练一致的代码、数据、路径、PyTorch/CUDA、设备和AMP配置。
训练结果及暂存文件在
`/kaggle/working`；它不是 Google Drive，运行时结束后不能保证保留。
按输出提示下载恢复 ZIP 或保存私有 Notebook 版本与输出后，再结束会话。
不要公开包含模型或原始标题的上传数据和输出。

模型任务是标题内实体的看多／中性／看空研究试验，资产身份、语义事件
和时序隔离尚未完成，开发集指标不代表证券预测或真实 X 效果。

In [ ]:
import os, sys, json, shutil, subprocess, importlib
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # 首次解析并显示提交；复现时替换成完整的 40 位提交 SHA。
PROJECT = Path("/kaggle/working/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF 不能唯一解析；请填写完整提交 SHA。")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/kaggle/working/130U-FinLLM 已属于其他仓库，请选择新目录。")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("本次固定代码提交：", REVISION)

# 保留 Colab 自带的 CUDA / PyTorch；此处仅安装项目的其他依赖。
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} 与现有环境不兼容：{incompatible}")
    print("tokenizers 兼容性检查通过：", installed)

def check_project_dependencies():
    # Read the refreshed installed metadata, not the previous editable checkout.
    problems = []
    for value in metadata.requires("finllm-130u") or []:
        requirement = Requirement(value)
        if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            problems.append(f"缺少 {requirement.name}")
            continue
        if installed not in requirement.specifier:
            problems.append(f"{requirement.name}=={installed} 不满足 {requirement}")
    if problems:
        raise RuntimeError("项目依赖检查失败：" + "; ".join(problems))
    print("项目已安装依赖检查通过。")

def activate_project_in_kernel():
    # pip installs the editable .pth for NEW Python processes. A notebook kernel
    # that was already running does not process that .pth again automatically.
    source = (PROJECT / "src").resolve()
    expected = source / "finllm"
    if not (expected / "__init__.py").is_file():
        raise RuntimeError(f"缺少项目 Python 源码：{expected}")
    sys.path[:] = [str(source)] + [entry for entry in sys.path if entry != str(source)]
    # Git may have changed code in the same directory since an earlier cell run.
    # Refresh only our modules; never reload Torch or its live CUDA state.
    for name in tuple(sys.modules):
        if name == "finllm" or name.startswith("finllm."):
            del sys.modules[name]
    importlib.invalidate_caches()
    package = importlib.import_module("finllm")
    runtime = importlib.import_module("finllm.runtime")
    for module, filename in ((package, "__init__.py"), (runtime, "runtime.py")):
        if Path(module.__file__).resolve() != expected / filename:
            raise RuntimeError(f"当前 Notebook 导入了错误的项目位置：{module.__file__}")
    if not callable(runtime.restore_seed_checkpoint):
        raise RuntimeError("当前 Notebook 的检查点恢复接口不可用。")
    print("当前 Notebook 内核导入检查通过：", runtime.__file__)

def check_fresh_process_import():
    # Verify the installed editable package separately, without injecting src
    # into PYTHONPATH. Both the current kernel and future CLI processes must work.
    probe = """import json, sys
from pathlib import Path
import finllm
import finllm.runtime as runtime
expected = Path(sys.argv[1]).resolve() / 'finllm'
for module, filename in ((finllm, '__init__.py'), (runtime, 'runtime.py')):
    if Path(module.__file__).resolve() != expected / filename:
        raise RuntimeError('Fresh process imported the wrong project: ' + str(module.__file__))
if not callable(runtime.restore_seed_checkpoint):
    raise RuntimeError('Fresh process checkpoint restore is unavailable')
print(json.dumps({'finllm': str(Path(finllm.__file__).resolve()), 'runtime': str(Path(runtime.__file__).resolve())}))
"""
    result = subprocess.check_output([sys.executable, "-c", probe, str(PROJECT / "src")],
                                     cwd=PROJECT, text=True)
    print("新 Python 进程导入检查通过：", result.strip())

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("需要 PyTorch >= 2.5；请更换较新的 Colab 运行环境。")
tokenizer_requirements = tokenizer_install_requirements()
print("本次 tokenizers 约束：", tokenizer_requirements)
torch_version_before = metadata.version("torch")
torch_file_before = Path(torch.__file__).resolve()
# One resolver transaction replaces the previous finllm metadata AND its
# dependencies. Upgrading tokenizers first leaves old finllm bounds active and
# produces a misleading conflict warning even when the eventual install works.
# The exact installed Torch build is constrained, so pip keeps Colab's CUDA build.
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".",
        "torch==" + torch_version_before, *tokenizer_requirements)
importlib.invalidate_caches()
if metadata.version("torch") != torch_version_before or Path(torch.__file__).resolve() != torch_file_before:
    raise RuntimeError("PyTorch 环境意外变化，请保留输出并停止训练。")
check_tokenizer_compatibility(tokenizer_requirements)
check_project_dependencies()
activate_project_in_kernel()
check_fresh_process_import()
print("Notebook 环境准备完成；PyTorch 保持：", torch_version_before)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("缺少阶段输入，请先准备或从 Drive 恢复：\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/kaggle/working")
print(f"/kaggle/working 剩余磁盘：{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
from finllm.kaggle_sft_runtime import (install_transfer_archive, install_transfer_directory,
    install_recovery_archive, install_recovery_directory)
from finllm.sft_pilot_runtime import prepare_sft_pilot, run_sft_pilot, backup_sft_pilot
from pathlib import Path

# 分别复制 04b 输出的转运清单 SHA256 与转运 ZIP SHA256。
TRANSFER_MANIFEST_SHA256 = "请粘贴04b输出的转运清单SHA256".strip().lower()
TRANSFER_ZIP_SHA256 = "请粘贴04b输出的转运ZIP的SHA256".strip().lower()
# 跨会话恢复时，把之前的恢复 ZIP 作为私有 Input 挂载并填写其 SHA。
# 首次运行保持空字符串；同一会话从本页重跑也不需要填写。
RECOVERY_MANIFEST_SHA256 = "".strip().lower()
RECOVERY_ZIP_SHA256 = "".strip().lower()
INPUT_ROOT = Path("/kaggle/input")
HANDOFF = Path("/kaggle/working/130u_handoff")
manifests = sorted(INPUT_ROOT.rglob("transfer_manifest.json"))
archives = sorted(INPUT_ROOT.rglob("130u_kaggle_transfer.zip"))
if len(manifests) == 1:
    transfer = install_transfer_directory(manifests[0].parent, HANDOFF,
                                          expected_manifest_sha256=TRANSFER_MANIFEST_SHA256)
elif not manifests and len(archives) == 1:
    transfer = install_transfer_archive(archives[0], HANDOFF, expected_zip_sha256=TRANSFER_ZIP_SHA256)
else:
    raise RuntimeError("请通过 Add Input 挂载唯一的私有转运数据集。")
print("原成果校验完成：", json.dumps(transfer, indent=2, ensure_ascii=False))
if RECOVERY_MANIFEST_SHA256 or RECOVERY_ZIP_SHA256:
    manifests = sorted(INPUT_ROOT.rglob("recovery_manifest.json"))
    candidates = sorted(INPUT_ROOT.rglob("130u_sft_recovery_*.zip"))
    if len(manifests) == 1:
        restored = install_recovery_directory(manifests[0].parent, HANDOFF,
                                 expected_manifest_sha256=RECOVERY_MANIFEST_SHA256)
    elif not manifests and len(candidates) == 1:
        restored = install_recovery_archive(candidates[0], HANDOFF,
                                 expected_zip_sha256=RECOVERY_ZIP_SHA256)
    else:
        raise RuntimeError("请挂载唯一的私有恢复数据集。")
    print("已恢复私有SFT进度：", json.dumps(restored, indent=2, ensure_ascii=False))

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("请在 Session options 选择 GPU T4，并从第一格重新运行。")
print("GPU数量：", torch.cuda.device_count(), "本次使用：", torch.cuda.get_device_name(0))
pilot = prepare_sft_pilot(PROJECT, HANDOFF, run_version="sft_kaggle_v1")
print("已核验领域母模型和实际排除池：", json.dumps(pilot["parent_audit"], indent=2, ensure_ascii=False))
print("任务分区：", json.dumps(pilot["counts"], indent=2, ensure_ascii=False))

In [ ]:
from finllm.kaggle_sft_runtime import export_recovery
from datetime import datetime, timezone
SFT_PROBE_SUCCEEDED = False
SFT_TRAIN_SUCCEEDED = False
SFT_EVAL_SUCCEEDED = False
probe = run_sft_pilot(pilot, "probe", finllm)
print(json.dumps(probe, indent=2, ensure_ascii=False))
SFT_PROBE_SUCCEEDED = True
recovery_file = Path("/kaggle/working") / ("130u_sft_recovery_probe_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".zip")
print("单步恢复包（须下载或保存版本才持久）：", json.dumps(export_recovery(HANDOFF, recovery_file), indent=2, ensure_ascii=False))

In [ ]:
SFT_TRAIN_SUCCEEDED = False
SFT_EVAL_SUCCEEDED = False
if not globals().get("SFT_PROBE_SUCCEEDED", False):
    raise RuntimeError("请先完成单步检查。")
result = run_sft_pilot(pilot, "continue", finllm)
print(json.dumps(result, indent=2, ensure_ascii=False))
SFT_TRAIN_SUCCEEDED = True

In [ ]:
SFT_EVAL_SUCCEEDED = False
if not globals().get("SFT_TRAIN_SUCCEEDED", False):
    raise RuntimeError("请先完成训练。")
output = Path(pilot["config"]["output_dir"])
development = pilot["task_local"] / "development.jsonl"
for name, checkpoint in (("domain_before", Path(pilot["config"]["init_checkpoint"])),
                         ("sft_best", output / "best.pt")):
    finllm("evaluate", "--checkpoint", checkpoint, "--tokenizer", pilot["config"]["tokenizer"],
           "--input", development, "--output", output / (name + "_development.json"),
           "--target-mode", "literal_entity")
    print(name, (output / (name + "_development.json")).read_text(encoding="utf-8"))
SFT_EVAL_SUCCEEDED = True

In [ ]:
if not globals().get("SFT_TRAIN_SUCCEEDED", False) or not globals().get("SFT_EVAL_SUCCEEDED", False):
    raise RuntimeError("请先完成训练与开发集比较。")
backup = backup_sft_pilot(pilot)
print("运行结果本地备份：", json.dumps(backup, indent=2, ensure_ascii=False))
recovery_file = Path("/kaggle/working") / ("130u_sft_recovery_final_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".zip")
print("最终恢复包（请下载或保存私有版本输出）：", json.dumps(export_recovery(HANDOFF, recovery_file), indent=2, ensure_ascii=False))

恢复包与转运包均含私有数据，只能保存在本人账户。保留恢复包 SHA、
Notebook 首格代码 SHA、开发集结果与备份输出。Kaggle 进程或会话被突然回收时，
未下载或保存的本地进度可能丢失；原领域权重仍在原 Drive。